In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, Activation, Add, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import ramanspy as rp
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

import os

In [2]:
# 1. Load RamanSPy Bacteria Dataset (train, val, test) from "data" folder
# -------------------------------------------------------------

dir_ = os.path.join(os.getcwd(), "data")

X_train_sc, y_train = rp.datasets.bacteria("train", folder=dir_)
X_val_sc,   y_val   = rp.datasets.bacteria("val", folder=dir_)
X_test_sc,  y_test  = rp.datasets.bacteria("test", folder=dir_)

# Labels for class names
y_labels, antibiotics_labels = rp.datasets.bacteria("labels")

# Print dataset sizes
print(f"Train: {len(y_train)} spectra")
print(f"Val:   {len(y_val)} spectra")
print(f"Test:  {len(y_test)} spectra")

Train: 60000 spectra
Val:   3000 spectra
Test:  3000 spectra


In [3]:
# Extract spectral data (NumPy arrays)
X_train = X_train_sc.spectral_data
X_val_orig   = X_val_sc.spectral_data
X_test  = X_test_sc.spectral_data

print("Original shapes:")
print("Train:", X_train.shape, "Val:", X_val_orig.shape, "Test:", X_test.shape)


Original shapes:
Train: (60000, 1000) Val: (3000, 1000) Test: (3000, 1000)


In [4]:
# 5. One-hot encode labels
# -------------------------------------------------------------
num_classes = len(y_labels)
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_cat   = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_cat  = tf.keras.utils.to_categorical(y_test, num_classes)

print("Label shape (one-hot):", y_train_cat.shape)

Label shape (one-hot): (60000, 30)


In [5]:
# --- 1. Define the Residual Block (The "Shortcut" part) ---
def residual_block(x, kernel_size, num_filters):
    """
    A single Residual Block (identity connection).
    The text describes a shortcut connection around ONE convolutional layer.
    """
    # 1. Store the input for the shortcut connection
    shortcut = x

    # 2. Convolutional Layer
    # Kernel number is 1 (num_filters=1), size is 3 (kernel_size=3)
    # Kaiming Normal initialization used (He initialization)
    x = Conv1D(
        filters=num_filters,
        kernel_size=kernel_size,
        padding='same',  # 'same' ensures input and output shape match
        kernel_initializer=HeNormal(),
        activation='relu'
    )(x)

    # 3. Add the shortcut to the output of the convolutional layer
    # This is the "residual" step: Output = F(x) + x
    x = Add()([x, shortcut])

    return x

In [6]:
# --- 2. Define the RamanNet Model ---

def create_ramannet(input_dim, num_classes):
    """
    Builds the RamanNet model:
        input_dim (int): M, the dimension of each spectrum (e.g., 1000).
        num_classes (int): N, the number of bacterial classes (e.g., 30).
    """
    # Input layer: The 1D spectrum data (Shape: M x 1 channel)
    inputs = Input(shape=(input_dim, 1), name='Input_Spectrum')

    # Initial Convolution Layer
    # Kernel size 7, Kernel number 1
    x = Conv1D(
        filters=1,
        kernel_size=7,
        padding='same',
        kernel_initializer=HeNormal(),
        activation='relu',
        name='Initial_Conv1D'
    )(inputs)

    # --- Two Residual Layers (Residual Blocks) ---

    # Residual Layer 1
    # Uses kernel size 3, kernel number 1 
    x = residual_block(x, kernel_size=3, num_filters=1)
    x = Activation('relu', name='Residual_Block_1_Output')(x) # ReLU after Add

    # Residual Layer 2
    # Uses kernel size 3, kernel number 1
    x = residual_block(x, kernel_size=3, num_filters=1)
    x = Activation('relu', name='Residual_Block_2_Output')(x) # ReLU after Add

    # Flatten Layer
    x = Flatten(name='Flatten_Features')(x)

    # Final Fully Connected Classification Layer
    # N output categories (30 for Bacteria-ID) with softmax for probabilities
    outputs = Dense(
        num_classes, 
        activation='softmax', 
        name='Final_Classification_Layer'
    )(x)

    # Create the functional model
    model = Model(inputs=inputs, outputs=outputs, name='RamanNet')

    return model

In [7]:
# --- 3. Instantiate and Compile the Model ---
M_DIMENSION = 1000  # M: Dimension of each spectrum
N_CLASSES = 30      # N: Bacterial classes (for the 30-isolate task)
LEARNING_RATE = 0.001
BATCH_SIZE = 10     # Used for training, not in the model definition
# Directory to save the best model from each fold
SAVE_DIR = 'ramannet_folds'
os.makedirs(SAVE_DIR, exist_ok=True)

In [8]:
# --- 1. Define K-Fold Parameters ---
N_SPLITS = 5
RANDOM_SEED = 42 # For reproducibility
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

# List to store the final test accuracy of each of the 5 models
all_test_accuracies = []

# --- 2. Start the Cross-Validation Loop ---
print(f"--- Starting {N_SPLITS}-Fold Cross-Validation for Fine-Tuning ---")

# We iterate 5 times, creating a new model for each split
for fold_n, (train_index, val_index) in enumerate(skf.split(X_val_orig, np.argmax(y_val_cat, axis=1))):   
    fold_label = fold_n + 1
    print(f"\n====================== FOLD {fold_label}/{N_SPLITS} ======================")
    
    # ------------------ A. Split the Data for This Fold ------------------
    # The K-Fold iterator gives us indices to split the X_val_orig pool into
    # a 90% training split (X_fine) and a 10% validation split (X_val)
    X_fine, X_val = X_val_orig[train_index], X_val_orig[val_index]
    y_fine, y_val = y_val_cat[train_index], y_val_cat[val_index]
    
    # ------------------ B. Re-create and Compile the Model ------------------
    # A fresh model instance must be created for each fold to ensure
    # that the training starts from the pre-trained state every time.
    ramannet_model = create_ramannet(input_dim=M_DIMENSION, num_classes=N_CLASSES)
    
    ramannet_model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # ------------------ C. Step 1: Pre-training (10 Epochs) ------------------
    print(f"[{fold_n + 1}] Pre-training on reference subset (10 epochs)...")
    ramannet_model.fit(
        X_train, y_train_cat,
        epochs=10, 
        batch_size=10, 
        verbose=1 # Set to 1 to see output per epoch
    )
    

    # ------------------ D. Step 2: Fine-Tuning (Max 30 Epochs) ------------------
    print(f"[{fold_label}] Fine-tuning on fold data (max 30 epochs with Early Stopping)...")
    
    checkpoint_filepath = os.path.join(SAVE_DIR, f'ramannet_fold_{fold_label}.keras')
    
    model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_best_only=True,         # Only saves when validation loss improves
        monitor='val_loss',          # Metric to monitor
        mode='min',                  # Save when val_loss decreases (minimum)
        verbose=0
    )
    
    
    
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=10,
        restore_best_weights=False, # We use ModelCheckpoint to save and restore later
        verbose=0 
    )

    ramannet_model.fit(
        X_fine, y_fine,
        epochs=30,  
        batch_size=10,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping, model_checkpoint_callback],
        verbose=1 
    )
    

# --- E. Step 3: Evaluate on Final Test Set ---
    # Load the best weights saved by ModelCheckpoint for evaluation
    best_model_for_fold = tf.keras.models.load_model(checkpoint_filepath)
    loss, accuracy = best_model_for_fold.evaluate(X_test, y_test_cat, verbose=0)
    
    print(f"[{fold_label}] Saved Best Model: {checkpoint_filepath}")
    print(f"[{fold_label}] Test Accuracy: {accuracy*100:.2f}% | Final Loss: {loss:.4f}")
    
    all_test_accuracies.append(accuracy)

--- Starting 5-Fold Cross-Validation for Fine-Tuning ---

====================== FOLD 1/5 ======================
[1] Pre-training on reference subset (10 epochs)...
Epoch 1/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.0315 - loss: 3.4013
Epoch 2/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.0305 - loss: 3.4013
Epoch 3/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0310 - loss: 3.4013
Epoch 4/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0313 - loss: 3.4013
Epoch 5/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0305 - loss: 3.4013
Epoch 6/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0316 - loss: 3.4013
Epoch 7/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0316 - loss: 3.4013
Epoch 8/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0321 - loss: 3.4013
Epoch 9/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.0308 - loss: 3.4013
Epoch 10/10
6000/6000 ━━━━━━━━━━━━

In [9]:
# --- 4. Report Final Average Accuracy ---
avg_accuracy = np.mean(all_test_accuracies)
std_accuracy = np.std(all_test_accuracies)

print("\n=======================================================")
print(f"Average Test Accuracy: {avg_accuracy*100:.2f}% ± {std_accuracy*100:.2f}%")
print("=======================================================")


Average Test Accuracy: 47.98% ± 36.70%


In [15]:
best_test_accuracy = -1.0
best_model_path_to_copy = ''



best_model_for_fold = tf.keras.models.load_model(checkpoint_filepath)
loss, accuracy = best_model_for_fold.evaluate(X_test, y_test_cat, verbose=0) 
    
print(f"[{fold_label}] Saved Best Weights for Fold: {checkpoint_filepath}")
print(f"[{fold_label}] Test Accuracy: {accuracy*100:.2f}% | Final Loss: {loss:.4f}")
    
    # ⭐ NEW LOGIC: Check if this fold is the overall best
if accuracy > best_test_accuracy:
    best_test_accuracy = accuracy
    best_model_path_to_copy = checkpoint_filepath

print("\n=======================================================")
print("Cross-Validation Complete. Saving Best Model...")

# Load the single best model found across all folds
best_model_to_save = tf.keras.models.load_model(best_model_path_to_copy)

# Define the final save path for the model
final_save_path = os.path.join(os.getcwd(), 'final_best_model.keras')

# Save the model to the user-requested common filename
best_model_to_save.save(final_save_path) 

print(f"Transfer Learning Model Saved:")
print(f"File: {final_save_path}")
print(f"Overall Best Test Accuracy: {best_test_accuracy*100:.2f}%")
print("=======================================================")

[5] Saved Best Weights for Fold: ramannet_folds\ramannet_fold_5.keras
[5] Test Accuracy: 83.43% | Final Loss: 0.6793

Cross-Validation Complete. Saving Best Model...
Transfer Learning Model Saved:
File: C:\Users\Rima\OneDrive - Instituto Politécnico de Lisboa\Exp dataset raman\Bacteria classifiers\final_best_model.keras
Overall Best Test Accuracy: 83.43%
